<a href="https://colab.research.google.com/github/samif0/orion/blob/main/experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Orion — Experiments Notebook

Runs and compares training/eval across attention backends: **dense**, **window**, and **sparse** (window + expander).

| Backend | Config |
|---|---|
| Dense | `configs/tinyshakespeare_dense.yaml` |
| Window | `configs/tinyshakespeare_window.yaml` |
| Sparse | `configs/tinyshakespeare_sparse.yaml` |

## 1. Setup

In [ ]:
import os, subprocess, sys

# Clone and install (Colab only — skip if running locally)
if not os.path.exists('orion'):
    subprocess.run(['git', 'clone', 'https://github.com/samif0/orion.git'], check=True)
    os.chdir('orion')
elif os.path.basename(os.getcwd()) != 'orion':
    os.chdir('orion')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt', '-r', 'requirements-dev.txt'], check=True)
print('Setup complete. cwd:', os.getcwd())

In [ ]:
import importlib.util, torch
print(f'torch {torch.__version__} | CUDA: {torch.cuda.is_available()} | device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}')

## 2. Config

In [ ]:
# Which experiments to run — set to False to skip
RUN_DENSE  = True
RUN_WINDOW = True
RUN_SPARSE = True

EXPERIMENTS = {
    'dense':  {'config': 'configs/tinyshakespeare_dense.yaml',  'run': RUN_DENSE},
    'window': {'config': 'configs/tinyshakespeare_window.yaml', 'run': RUN_WINDOW},
    'sparse': {'config': 'configs/tinyshakespeare_sparse.yaml', 'run': RUN_SPARSE},
}

## 3. Training

In [ ]:
import subprocess, time

train_results = {}

for name, exp in EXPERIMENTS.items():
    if not exp['run']:
        print(f'[{name}] skipped')
        continue
    print(f'\n=== Training: {name} ({exp["config"]}) ===')
    t0 = time.time()
    result = subprocess.run(
        [sys.executable, '-m', 'orion', 'train', '--config', exp['config']],
        capture_output=False
    )
    elapsed = time.time() - t0
    train_results[name] = {'returncode': result.returncode, 'elapsed_s': round(elapsed, 1)}
    status = '✅' if result.returncode == 0 else '❌'
    print(f'{status} {name} finished in {elapsed:.1f}s (rc={result.returncode})')

print('\nTrain summary:', train_results)

## 4. Evaluation

In [ ]:
import glob

eval_results = {}

for name, exp in EXPERIMENTS.items():
    if not exp['run']:
        continue
    # Find checkpoint
    checkpoints = glob.glob(f'runs/**/*.pt', recursive=True)
    # Pick the most recently modified one for this run
    run_dir = None
    import yaml
    with open(exp['config']) as f:
        cfg = yaml.safe_load(f)
    out_dir = cfg.get('run', {}).get('out_dir', 'runs/latest')
    ckpt = f'{out_dir}/checkpoint.pt'

    if not os.path.exists(ckpt):
        print(f'[{name}] No checkpoint found at {ckpt}, skipping eval')
        continue

    print(f'\n=== Eval: {name} (checkpoint: {ckpt}) ===')
    result = subprocess.run(
        [sys.executable, '-m', 'orion', 'eval', '--config', exp['config'], '--checkpoint', ckpt],
        capture_output=True, text=True
    )
    eval_results[name] = {'stdout': result.stdout, 'returncode': result.returncode}
    print(result.stdout)
    if result.returncode != 0:
        print('STDERR:', result.stderr)

## 5. Compare Metrics

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

metrics = {}

for name, exp in EXPERIMENTS.items():
    if not exp['run']:
        continue
    with open(exp['config']) as f:
        cfg = yaml.safe_load(f)
    out_dir = cfg.get('run', {}).get('out_dir', 'runs/latest')
    jsonl_path = f'{out_dir}/metrics.jsonl'
    if not os.path.exists(jsonl_path):
        print(f'[{name}] No metrics file at {jsonl_path}')
        continue
    rows = []
    with open(jsonl_path) as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    metrics[name] = pd.DataFrame(rows)
    print(f'[{name}] {len(rows)} steps logged')

# Plot training loss
if metrics:
    fig, ax = plt.subplots(figsize=(10, 5))
    for name, df in metrics.items():
        if 'loss' in df.columns and 'step' in df.columns:
            ax.plot(df['step'], df['loss'], label=name)
    ax.set_xlabel('Step')
    ax.set_ylabel('Loss')
    ax.set_title('Training Loss by Attention Backend')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('runs/loss_comparison.png', dpi=150)
    plt.show()
    print('Saved: runs/loss_comparison.png')

In [ ]:
# Summary table: final loss + training time
rows = []
for name, df in metrics.items():
    final_loss = df['loss'].iloc[-1] if 'loss' in df.columns else None
    elapsed = train_results.get(name, {}).get('elapsed_s')
    rows.append({'backend': name, 'final_loss': round(final_loss, 4) if final_loss else 'N/A', 'train_time_s': elapsed})

summary = pd.DataFrame(rows).set_index('backend')
print(summary.to_string())

## 6. Custom Run

Run a one-off experiment with custom parameters.

In [ ]:
# Edit these and run the cell
CUSTOM_CONFIG = 'configs/tinyshakespeare_sparse.yaml'
EXTRA_ARGS = []  # e.g. ['--steps', '500']

print(f'Running: orion train --config {CUSTOM_CONFIG} {" ".join(EXTRA_ARGS)}')
subprocess.run([sys.executable, '-m', 'orion', 'train', '--config', CUSTOM_CONFIG] + EXTRA_ARGS)